# 🧠 Memory in LangChain

## The Problem: LLMs Are Stateless
Each LLM call is **independent**. The LLM has no memory of previous messages.

```
Call 1: "My name is Alice"  → "Hello Alice!"
Call 2: "What's my name?"   → "I don't know your name."  ← !!
```

## The Solution: Memory
We **manually** manage conversation history and inject it into every prompt.

## Memory Types
| Type | What It Stores | Use Case |
|------|---------------|---------|
| ConversationBufferMemory | Full history | Short conversations |
| ConversationBufferWindowMemory | Last K messages | Medium conversations |
| ConversationSummaryMemory | LLM-compressed summary | Long conversations |
| ConversationSummaryBufferMemory | Summary + recent messages | Best of both worlds |
| VectorStoreRetrieverMemory | Semantic search over history | Very long history |

## Modern Approach: LangGraph
The old LangChain memory classes are being **replaced** by LangGraph's checkpointers.
We teach both: legacy for understanding, LangGraph for production.


In [ ]:
# ── Manual Memory: The Foundation ──────────────────────────────────────
# Before any memory class, understand what's happening manually.
# Memory = just maintaining and injecting a list of messages.

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# Our 'memory' is just a list
conversation_history: list = [
    SystemMessage(content='You are a helpful assistant.')
]

def chat(user_input: str) -> str:
    # Add user message to history
    conversation_history.append(HumanMessage(content=user_input))

    # Send FULL history to LLM
    response = llm.invoke(conversation_history)

    # Add AI response to history
    conversation_history.append(response)  # AIMessage

    return response.content

# Simulate conversation
print('Turn 1:', chat('My name is Alice and I love Python.'))
print('Turn 2:', chat('What is my name and what do I love?'))
print(f'\nHistory length: {len(conversation_history)} messages')

In [ ]:
# ── RunnableWithMessageHistory (Modern LCEL approach) ───────────────────
# WHY: Wraps any LCEL chain to automatically manage message history.
# Uses session_id to separate conversations from different users.

from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

# Storage for all sessions
store: dict[str, InMemoryChatMessageHistory] = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# Base chain (no memory yet)
prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant.'),
    MessagesPlaceholder(variable_name='history'),  # ← memory injected here
    ('human', '{input}'),
])

chain = prompt | ChatOpenAI(model='gpt-4o-mini') | StrOutputParser()

# Wrap with memory
chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key='input',
    history_messages_key='history',
)

# Different users, different sessions
config_alice = {'configurable': {'session_id': 'alice_session'}}
config_bob   = {'configurable': {'session_id': 'bob_session'}}

r1 = chain_with_memory.invoke({'input': 'My name is Alice.'}, config=config_alice)
r2 = chain_with_memory.invoke({'input': 'My name is Bob.'}, config=config_bob)
r3 = chain_with_memory.invoke({'input': 'What is my name?'}, config=config_alice)
r4 = chain_with_memory.invoke({'input': 'What is my name?'}, config=config_bob)

print('Alice asks what her name is:', r3)
print('Bob asks what his name is:', r4)

## 🎯 Interview Questions

1. **Why are LLMs stateless and how do we add memory?**
2. **What is RunnableWithMessageHistory?**
3. **What is a session_id and why is it needed?**
4. **Difference between ConversationBufferMemory and ConversationSummaryMemory?**
5. **How does LangGraph improve on legacy memory classes?**

> Answer these before moving to the next module.

## 💪 Mini Exercise

**Exercise**: Based on what you learned in this module:

1. Re-run all code cells with your own inputs
2. Modify one code cell to add new functionality
3. Answer the interview questions above from memory

**Mini Project**: Build a small application that combines the concepts from this module.


## 📚 Summary

In this module you learned:
- The core concepts of **Memory — Giving LLMs a Past**
- How to implement them with modern LangChain/LangGraph APIs
- Common mistakes and how to avoid them
- Interview-level understanding of why each component exists

**Next**: Continue to the next module to build on these foundations.

---
*Part of the LangChain & LangGraph Master Course*
